# Full-sheet detection and graph recovery

Detect every symbol, junction and text label on a single-line diagram with YOLOv8, then trace the wires to recover the electrical graph (nodes, power / measurement edges, nets). Pipeline after AITEE and CircuitNet; code lives in `src/Object Detection/SLD Training/`.

**Before running**: GPU on, Internet on, attach the `sld-sheets` dataset built by `build_sld_dataset.py` (images, labels, graphs, data.yaml).

In [ ]:
# imports
import json, os, shutil, subprocess, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml
from PIL import Image

## Setup

In [ ]:
REPO_URL = "https://github.com/ishuu19/Single-Line-Diagram-and-Energy-Mapping.git"
REPO = Path("/kaggle/working/Single-Line-Diagram-and-Energy-Mapping")
TRAIN_DIR = REPO / "src" / "Object Detection" / "SLD Training"
SAVE_DIR = Path("/kaggle/working/sld_outputs")
INPUT = Path("/kaggle/input")

In [ ]:
def sh(*cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    return (r.stdout + r.stderr).strip()


def find_dataset(marker="data.yaml"):
    for p in INPUT.rglob(marker):
        if (p.parent / "images" / "train").is_dir():
            return p.parent
    raise FileNotFoundError("sld-sheets dataset not attached")


def link(src, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_symlink() or dest.exists():
        dest.unlink() if dest.is_symlink() else shutil.rmtree(dest)
    os.symlink(src, dest, target_is_directory=True)

In [ ]:
if (REPO / ".git").is_dir():
    sh("git", "pull", "--ff-only", cwd=REPO)
else:
    sh("git", "clone", "-q", REPO_URL, str(REPO))
assert (TRAIN_DIR / "train_sld.py").exists()

In [ ]:
os.chdir(TRAIN_DIR)
sys.path.insert(0, str(TRAIN_DIR))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "scikit-image", "networkx"], check=True)

In [ ]:
import config
from sld_data import load_classes, sheet_ids, image_path, load_graph, gt_detections
from wires import trace
from graph_eval import score_sheet, summarise, oracle_prediction
from train_sld import train
from predict_graph import load_detector, predict_sheet
from viz import show_graph

In [ ]:
SHEETS = find_dataset()
link(SHEETS, config.SHEETS_DIR)
classes = load_classes()
len(classes), {s: len(sheet_ids(s)) for s in ("train", "val", "test")}

## Data check

In [ ]:
sid = sheet_ids("val")[0]
graph = load_graph(sid)
image = Image.open(image_path(sid, "val"))
show_graph(image, {"symbols": gt_detections(graph, classes), "edges": []}, f"{sid}: ground-truth boxes");

## Wire tracer with ground-truth boxes
Scores the graph recovery on its own, before any detector error.

In [ ]:
rows = [score_sheet(*oracle_prediction(s, "val", classes)) for s in sheet_ids("val")[:20]]
summarise(rows)

In [ ]:
graph, result = oracle_prediction(sid, "val", classes)
show_graph(image, result, f"{sid}: recovered graph (red power, blue measurement)");

## Train the detector
YOLOv8s at 1280 px, no flips (symbols are orientation-specific). Set `EPOCHS = 2` for a smoke run.

In [ ]:
EPOCHS = 60
WEIGHTS = train(epochs=EPOCHS, project=SAVE_DIR / "runs")
WEIGHTS

In [ ]:
Image.open(SAVE_DIR / "runs" / "sld" / "results.png")

## Detector metrics on the test split

In [ ]:
from ultralytics import YOLO
metrics = YOLO(str(WEIGHTS)).val(data=str(SAVE_DIR / "runs" / "data.yaml"), split="test", imgsz=config.IMG_SIZE, plots=False, verbose=False)
{"mAP50": round(float(metrics.box.map50), 3), "mAP50-95": round(float(metrics.box.map), 3)}

## Detector + tracer: full graph recovery on the test split

In [ ]:
model = load_detector(WEIGHTS)
test_ids = sheet_ids("test")
rows = []
for s in test_ids:
    result, _ = predict_sheet(model, Image.open(image_path(s, "test")), classes, s)
    rows.append(score_sheet(load_graph(s), result))
summary = summarise(rows)
summary

In [ ]:
s = test_ids[0]
result, pred_graph = predict_sheet(model, Image.open(image_path(s, "test")), classes, s)
show_graph(Image.open(image_path(s, "test")), result, f"{s}: detected + traced");

In [ ]:
pred_graph["edges"][:5]

## Save

In [ ]:
config.OUTPUT_DIR.mkdir(exist_ok=True)
(config.OUTPUT_DIR / "graph_pred_test.json").write_text(json.dumps({"summary": summary, "sheets": dict(zip(test_ids, rows))}, indent=1))
for name in ("models", "outputs"):
    if (TRAIN_DIR / name).exists():
        shutil.copytree(TRAIN_DIR / name, SAVE_DIR / name, dirs_exist_ok=True)
sorted(p.name for p in (SAVE_DIR / "models").glob("*"))